In [7]:
# ======================================================================
# Connect to PostgreSQL and get the latest Bronze observation date
# ======================================================================

# Import libraries
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from pathlib import Path
import polars as pl

# Define project paths and load local database credentials from the .env file
PROJECT_PATH = Path(
    r"C:\Users\zychl\Documents\GitHub\data-toolkit\weather_data_processing"
)

load_dotenv(PROJECT_PATH / ".env") 

# PostgreSQL connection details
username = os.getenv('POSTGRES_USER')
password = os.getenv('POSTGRES_PASSWORD')
host = 'localhost'
port = os.getenv('HOST_POSTGRES_PORT')
database = os.getenv('POSTGRES_DB')

engine = create_engine(
    f'postgresql+pg8000://{username}:{password}@{host}:{port}/{database}'
    )

print('Connected.')

# Fetch the data from PostgreSQL analytics_db server - bronze layer
print('Executing query...')

bronze_max_date = pl.read_database(
    query=
    'SELECT MAX(observation_date) AS max_observation_date ' \
    'FROM bronze.weather',
    connection=engine
    )

bronze_max_date = bronze_max_date['max_observation_date'][0]

print(f'Executing query finished. The result is {bronze_max_date}')

Connected.
Executing query...
Executing query finished. The result is 2026-07-16


In [ ]:
# ======================================================================
# Get and prepare the latest weather data from NOAA
# ======================================================================

from datetime import datetime

current_year = datetime.now().year

url = f"https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/{current_year}.csv.gz"

print(f"Downloading NOAA weather data for {current_year}...")

noaa_weather_raw = pl.read_csv(
    url,
    has_header=False
    )

print("NOAA weather data downloaded.")

# Rename source columns
print("Renaming NOAA weather columns...")

noaa_weather = noaa_weather_raw.rename({
    "column_1": "station",
    "column_2": "observation_date",
    "column_3": "metric",
    "column_4": "value",
    "column_5": "measurement_flag",
    "column_6": "quality_flag",
    "column_7": "source_flag",
    "column_8": "observation_time"
})

# Convert observation_date to Date
print("Converting observation_date to Date...")

noaa_weather = noaa_weather.with_columns(
    pl.col('observation_date')
    .cast(pl.String)
    .str.strptime(pl.Date, '%Y%m%d')
)

print("NOAA weather data preparation completed.")

In [ ]:
# ======================================================================
# Filter new NOAA observations and load the incremental batch to Staging
# ======================================================================

print("Filtering new weather observations...")

print(f"Incremental batch contains {noaa_weather.height} rows.")

print("Exporting incremental batch...")

noaa_weather.write_database(
    table_name='weather_test',
    connection='engine',
    schema='staging',
    if_table_exists='append'
)

print("Incremental batch exported to bronze.weather")